# 02 · Delimitación de Población y Definición del Objetivo

## Marco Regulatorio: Condicionalidad al Default
Bajo la normativa **IFRS 9** (Fase 3) y el marco de **Basilea (IRB Avanzado)**:
* **La LGD se modela estrictamente condicionada a la ocurrencia de incumplimiento**:

$$E[LGD \mid \text{Default} = 1]$$

Los contratos no fallidos (`event == 0`) no ingresan al proceso judicial o extrajudicial de cobranza y no generan flujos de recuperación observables. Incluirlos subestimaría artificialmente el quebranto esperado.

## Protocolo de Prevención de Fuga de Información (*Data Leakage*)
El dataset incluye variables calculadas con posterioridad al default:
1. `Recovery_rate`: Complemento algebraico exacto ($1 - LGD$).
2. `y_logistic`, `lnrr`, `Y_probit`: Transformaciones matemáticas derivadas del resultado final.
3. `event`: Flag indicador que delimita la población.

Este cuaderno ejecuta:
1. Restricción muestral a `event == 1`.
2. Aislamiento estricto de variables predictoras disponibles al default.
3. Análisis de impacto de registros duplicados.
4. Partición estratificada 80/20.
5. Guardado de los conjuntos de datos en `data/processed/`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path("../src").resolve()))
from preprocessing import filter_defaulted_population, LEAKAGE_COLUMNS, TARGET

plt.style.use('seaborn-v0_8-whitegrid')
DATA_PATH = Path("../data/raw/lgd.csv")
df_raw = pd.read_csv(DATA_PATH)
print(f"Dimensiones del dataset crudo: {df_raw.shape}")

Dimensiones del dataset crudo: (2545, 8)


In [2]:
# Filtrado de Población: event == 1
print("--- Distribución del Estado de Evento ---")
display(df_raw['event'].value_counts(dropna=False).rename("Recuento").to_frame())

df_defaulted = filter_defaulted_population(df_raw)
print(f"Población en default retenida: {len(df_defaulted):,} contratos ({len(df_defaulted)/len(df_raw):.1%} del total).")
print(f"Contratos sin default excluidos: {len(df_raw) - len(df_defaulted):,} contratos.")

--- Distribución del Estado de Evento ---


,Recuento
event,
1,1817
0,728


Población en default retenida: 1,817 contratos (71.4% del total).
Contratos sin default excluidos: 728 contratos.


In [3]:
# Distribución de la LGD en la población en default
target_stats = df_defaulted[TARGET].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
display(target_stats.to_frame(name="LGD (Condicionada al Default)"))

print(f"Rango de LGD: [{df_defaulted[TARGET].min():.5f}, {df_defaulted[TARGET].max():.5f}]")
print(f"LGD <= 1%: {(df_defaulted[TARGET] < 0.01).sum():,} contratos ({(df_defaulted[TARGET] < 0.01).mean():.1%})")
print(f"LGD >= 95%: {(df_defaulted[TARGET] >= 0.95).sum():,} contratos ({(df_defaulted[TARGET] >= 0.95).mean():.1%})")

,LGD (Condicionada al Default)
count,1817.000000
mean,0.319529
std,0.350019
min,0.000018
5%,0.003223
25%,0.023072
50%,0.145219
75%,0.561990
95%,0.999990
max,0.999990


Rango de LGD: [0.00002, 0.99999]
LGD <= 1%: 245 contratos (13.5%)
LGD >= 95%: 203 contratos (11.2%)


In [4]:
# Matriz de Auditoría Antifuga
leakage_audit = pd.DataFrame([
    {"Variable": "LTV", "Rol": "Predictor Central de Colateral", "Estado": "PERMITIDO", "Disponible_en_Default": "Sí"},
    {"Variable": "purpose1", "Rol": "Destino del Inmueble", "Estado": "PERMITIDO", "Disponible_en_Default": "Sí"},
    {"Variable": "Recovery_rate", "Rol": "Complemento (1 - LGD)", "Estado": "PROHIBIDO (Fuga)", "Disponible_en_Default": "Post-Cobranza"},
    {"Variable": "y_logistic", "Rol": "Transformación Logística", "Estado": "PROHIBIDO (Fuga)", "Disponible_en_Default": "Post-Cobranza"},
    {"Variable": "lnrr", "Rol": "Logaritmo de Recuperación", "Estado": "PROHIBIDO (Fuga)", "Disponible_en_Default": "Post-Cobranza"},
    {"Variable": "Y_probit", "Rol": "Transformación Probit", "Estado": "PROHIBIDO (Fuga)", "Disponible_en_Default": "Post-Cobranza"},
    {"Variable": "event", "Rol": "Filtro de Población", "Estado": "EXCLUIDO de X", "Disponible_en_Default": "Filtro"}
])
display(leakage_audit)

,Variable,Rol,Estado,Disponible_en_Default
0,LTV,Predictor Central de Colateral,PERMITIDO,Sí
1,purpose1,Destino del Inmueble,PERMITIDO,Sí
2,Recovery_rate,Complemento (1 - LGD),PROHIBIDO (Fuga),Post-Cobranza
3,y_logistic,Transformación Logística,PROHIBIDO (Fuga),Post-Cobranza
4,lnrr,Logaritmo de Recuperación,PROHIBIDO (Fuga),Post-Cobranza
5,Y_probit,Transformación Probit,PROHIBIDO (Fuga),Post-Cobranza
6,event,Filtro de Población,EXCLUIDO de X,Filtro


In [5]:
# Análisis de registros duplicados
n_duplicados = df_defaulted.duplicated().sum()
print(f"Registros idénticos en la muestra en default: {n_duplicados} ({n_duplicados/len(df_defaulted):.2%})")

media_con = df_defaulted[TARGET].mean()
media_sin = df_defaulted.drop_duplicates()[TARGET].mean()
print(f"LGD media con duplicados:    {media_con:.4f}")
print(f"LGD media sin duplicados:    {media_sin:.4f} (Diferencia: {abs(media_con - media_sin):.4f})")
print("Conclusión: Los duplicados no alteran la media de forma material; se retienen para el split base.")

Registros idénticos en la muestra en default: 17 (0.94%)
LGD media con duplicados:    0.3195
LGD media sin duplicados:    0.3137 (Diferencia: 0.0059)
Conclusión: Los duplicados no alteran la media de forma material; se retienen para el split base.


In [6]:
# Partición Estratificada Train/Test (80/20 sobre purpose1)
FEATURES = ['LTV', 'purpose1']

X = df_defaulted[FEATURES]
y = df_defaulted[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=df_defaulted['purpose1']
)

print(f"Observaciones en Entrenamiento: {len(X_train):,} ({len(X_train)/len(df_defaulted):.0%})")
print(f"Observaciones en Validación:    {len(X_test):,} ({len(X_test)/len(df_defaulted):.0%})")
print(f"LGD Media Train: {y_train.mean():.4f} | LGD Media Test: {y_test.mean():.4f}")
print(f"Proporción de Alquiler Train: {X_train['purpose1'].mean():.2%} | Test: {X_test['purpose1'].mean():.2%}")

Observaciones en Entrenamiento: 1,453 (80%)
Observaciones en Validación:    364 (20%)
LGD Media Train: 0.3172 | LGD Media Test: 0.3289
Proporción de Alquiler Train: 8.88% | Test: 8.79%


In [7]:
# Guardar datasets procesados
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv(processed_dir / "lgd_train.csv", index=False)
test_df.to_csv(processed_dir / "lgd_test.csv", index=False)
print(f"Particiones guardadas en {processed_dir.resolve()}.")

Particiones guardadas en C:\Users\jhon_\Documents\GitHub\IFRS9-Credit-Risk\LGD\data\processed.


## Resumen y Próximo Paso
* Población restringida a contratos en default (`event == 1`, 1.817 observaciones).
* Fuga de datos purgada exhaustivamente.
* Partición estratificada guardada (1.453 train / 364 test).
* **Próximo Cuaderno**: [03_feature_engineering.ipynb](file:///c:/Users/jhon_/Documents/GitHub/IFRS9-Credit-Risk/LGD/notebooks/03_feature_engineering.ipynb) construye las transformaciones no lineales del colateral.